This notebook aims at implementing QAOA for comparison with L-VQE (Max-Cut, for 3-regular graphs). Based on one of the course homeworks.

In [1]:
from typing import List, Tuple

import networkx as nx
import pandas as pd
import pennylane as qml
import seaborn as sns
from matplotlib import pyplot as plt
from pennylane import numpy as np
from tqdm import tqdm
from pennylane.operation import Operation
import random

num_nodes = n_wires = 42

SEED=562    # for reproducibility
np.random.seed(SEED)
sns.set_theme()

ModuleNotFoundError: No module named 'pennylane'

In [ ]:
# Helper functions
def flatten(xss):
    return [x for xs in xss for x in xs]

def bitstring_to_int(bit_string_sample):
    bit_string = "".join(str(bs) for bs in bit_string_sample)
    return int(bit_string, base=2)

In [ ]:
def get_random_graph(N: int, seed: int) -> nx.Graph:
    assert N % 2 == 0
    rng_graph = np.random.RandomState(seed)
    while True:
        G = nx.random_regular_graph(3, N, seed=rng_graph)
        if nx.is_connected(G):
            return G

G = get_random_graph(num_nodes, seed=SEED)

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=SEED)
nx.draw(G, pos, with_labels=True, node_color='lightblue',
    node_size=400, font_size=10, font_weight='bold')
plt.title(f"Random regular graph G(k=3, N={G.number_of_nodes()}): {G.number_of_edges()} edges")
plt.show()

In [ ]:
dev = qml.device("lightning.qubit", wires=n_wires, seed=SEED)

In [ ]:
def mixer(beta: float):
    for j in range(n_wires):
        qml.RX(2*beta, wires=j)

def cost(gamma: float):
    for (qubit_a, qubit_b, weight) in G.edges():
        qml.CNOT(wires=(qubit_a, qubit_b))
        angle = -gamma * weight
        qml.RZ(angle, wires=qubit_b)
        qml.CNOT(wires=(qubit_a, qubit_b))

In [ ]:
@qml.set_shots(1000)
@qml.qnode(dev)
def qaoa_circuit(gammas, betas, edge, n_layers=1):

    # 1. Prepare the uniform superposition state
    for i in range(n_wires):
        qml.Hadamard(wires=i)

    # 2. Alternate n_layers times between U_C and U_M
    for p in range(n_layers):
        cost(gammas[p])
        mixer(betas[p])

    H = qml.PauliZ(edge[0]) @ qml.PauliZ(edge[1])
    return qml.expval(H)

In [ ]:
def cut_value(partition: int):
    bitstring = format(partition, f'0{n_wires}b')
    cut_val = 0
    for (j, k, weight) in G.edges():
        if bitstring[j] != bitstring[k]:
            cut_val += weight
    return cut_val

In [ ]:
def objective(params):
    gammas = params[0]
    betas = params[1]
    neg_obj = 0
    for (j, k, weight) in G.edges():
        # objective for the weighted MaxCut problem
        neg_obj -= 0.5 * weight * (1 - qaoa_circuit(gammas, betas, edge=(j, k)))
    return neg_obj

def qaoa_maxcut(
    n_layers: int = 1, opt=None, steps: int = 100
):
    print("\np={:d}".format(n_layers))
    if opt is None:
        opt=qml.AdagradOptimizer(stepsize=0.3)

    # optimize parameters in objective
    result_dict = {"step": [], "objective": [], "layers": []}

    # Sample some parameters
    params = 0.01 * np.random.rand(2, n_layers, requires_grad=True)

    for i in tqdm(range(steps)):
        params = opt.step(objective, params)
        obj_value = -objective(params)
        result_dict["step"].append(i + 1)
        result_dict["objective"].append(obj_value)
        result_dict["layers"].append(n_layers)

    # sample measured bitstrings 100 times
    bit_strings = []
    n_samples = 100
    for i in range(0, n_samples):
        bit_strings.append(
            bitstring_to_int(
                qaoa_circuit(params[0], params[1], edge=None)[0]
            )
        )

    # print optimal parameters and most frequently sampled bitstring
    counts = np.bincount(np.array(bit_strings))
    most_freq_bit_string = np.argmax(counts)
    print("Optimized (gamma, beta) vectors:\n{}".format(params[:, :n_layers]))
    print(
        "Most frequently sampled bit string is:",
        format(most_freq_bit_string, f"0{n_wires}b"),
    )

    return (-objective(params=params), bit_strings, result_dict, params[0], params[1])

In [ ]:
np.random.seed(seed=SEED)
random.seed(SEED)
obj1, bitstrings1, res1, final_gammas1, final_betas1 = qaoa_maxcut(n_layers=1, steps=50)
obj2, bitstrings2, res2, final_gammas2, final_betas2 = qaoa_maxcut(n_layers=2, steps=50)

In [ ]:
df = pd.concat([pd.DataFrame.from_dict(res1), pd.DataFrame.from_dict(res2)])
sns.lineplot(
    data=df, x="step", y="objective", hue="layers", style="layers", markers=True
)
plt.xticks(sorted([x for x in set(df["step"]) if x % 10 == 0]))
plt.show()

In [ ]:
xticks = range(0, 32)
xtick_labels = list(map(lambda x: format(x, f"0{n_wires}b"), xticks))
bins = np.arange(0, 33) - 0.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4))
plt.subplot(1, 2, 1)
plt.title("n_layers=1")
plt.xlabel("bitstrings")
plt.ylabel("freq.")
plt.xticks(xticks, xtick_labels, rotation="vertical")
plt.hist(bitstrings1, bins=bins)
plt.subplot(1, 2, 2)
plt.title("n_layers=2")
plt.xlabel("bitstrings")
plt.ylabel("freq.")
plt.xticks(xticks, xtick_labels, rotation="vertical")
plt.hist(bitstrings2, bins=bins)
# plt.tight_layout()
plt.show()